# Работа №2. RAG чат-бот
## Текст задания

Задача:
Создать базу знаний (тексты ≤1500 символов).
Подключить GigaChat API или Yandex Cloud AI Studio.
Подключить к Telegram.
Бот должен искать информацию и давать осмысленные ответы.

Результаты:
Код чат-бота (.ipynb).
База данных.
Скриншоты диалога.

Критерии:
Бот отвечает корректно по базе.
Функциональность Telegram.

Сдача:
Все файлы в облако, ссылка в комментариях.

## Выполнение
Для создания бота в телеграм воспользуемся @BotFather, чтобы задать его идентификаторы. А чтобы использовать большую языковую модель необходимо сначала достать ключ GigaChat. На платформме указано, что сам ключ действителен всего 30 минут.

In [4]:
!pip install langchain langchain-text-splitters langchain-community gigachat sentence-transformers faiss-cpu python-telegram-bot nest_asyncio

### Создание бзы знаний
Создадим текстовый файл с информацией, по которой бот будет отвечать. Пусть это будет тема "Система умного дома", разбив текст на блоки менее 1500 символов

In [2]:
data = """
ОБЩИЕ СВЕДЕНИЯ И ХАБЫ
Экосистема SmartLife поддерживает три протокола связи: Wi-Fi (2.4 ГГц), Zigbee 3.0 и Bluetooth Mesh.
Центральный контроллер (Хаб) является обязательным устройством для работы датчиков Zigbee.
Максимальное количество устройств, подключаемых к одному Хабу — 128 единиц.
Световой индикатор Хаба: синий мигающий — режим сопряжения, красный — потеря сети, зеленый — штатная работа.

ДАТЧИКИ БЕЗОПАСНОСТИ И ОСВЕЩЕНИЯ
1. Датчик протечки воды: Питается от батарейки CR2032. Срок службы батареи — до 2 лет. При срабатывании отправляет PUSH-уведомление и перекрывает краны (если установлен электропривод).
2. Датчик движения: Угол обзора 170 градусов, дистанция обнаружения до 7 метров. Имеет встроенный датчик освещенности (люксметр).
3. Умные лампы: Поддерживают регулировку цветовой температуры от 2700K (теплый) до 6500K (холодный). Мощность 9Вт, световой поток 800 лм.

СЦЕНАРИИ И АВТОМАТИЗАЦИЯ
Сценарии делятся на "Автоматизации" (выполняются по условию) и "Запуски нажатием" (виджеты в телефоне).
Пример автоматизации "Ушел из дома": Если датчик на двери размыкается более 2 минут и движения в коридоре нет, то выключить весь свет, перевести кондиционер в режим эко и активировать режим охраны.
Задержка выполнения (Delay): В сценариях можно выставлять паузу от 1 секунды до 5 часов между действиями.

УПРАВЛЕНИЕ КЛИМАТОМ
Умный термостат работает с газовыми и электрическими котлами (сухой контакт).
Диапазон регулировки температуры: от +5 до +35 градусов Цельсия. Погрешность составляет 0.5 градуса.
Режим "Антизамерзание": Автоматически включает нагрев, если температура в помещении падает ниже +7 градусов, чтобы предотвратить разрыв труб.

ТЕХНИЧЕСКАЯ ПОДДЕРЖКА И УСТРАНЕНИЕ ОШИБОК
Ошибка E1: Потеря связи с сервером. Проверьте настройки роутера (порт 8883 должен быть открыт).
Ошибка E5: Низкий заряд батареи на конечном устройстве (менее 10%).
Сброс настроек устройства (Hard Reset): Удерживайте кнопку сопряжения на корпусе в течение 10 секунд до быстрого мигания индикатора.
При смене пароля Wi-Fi все Wi-Fi устройства нужно переподключать заново, устройства Zigbee сохраняют привязку к Хабу.
"""

with open("data_base.txt", "w", encoding="utf-8") as f:
    f.write(data)

### Настройка бота

In [6]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

Для создания эмбеддингов используем бесплатную модель от HuggingFace

In [7]:
loader = TextLoader("data_base.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

/tmp/ipykernel_4633/575480926.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or dat

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Логика чат-бота и интеграция с Telegram
В этой ячейке нужно вставить свои токены. Код использует nest_asyncio, чтобы Telegram бот корректно работал внутри среды Colab.

In [8]:
import asyncio
import nest_asyncio
from telegram import Update
from telegram.ext import ApplicationBuilder, ContextTypes, MessageHandler, filters
from langchain_community.chat_models import GigaChat

In [9]:
nest_asyncio.apply()
GIGACHAT_CREDENTIALS = "NWQyOWFjZTMtZmVlMy00YjhmLWIzODEtMWE0N2VkNWMzY2FlOjJiMjQwMjE4LTc3MDAtNDA2Zi1iOGJkLTI0N2I0MWI5YTQ3ZQ=="
TELEGRAM_TOKEN = "8618563666:AAGQ29228F9y3eQhDn_OEHjzxyLrBMJcA1s"

llm = GigaChat(credentials=GIGACHAT_CREDENTIALS, verify_ssl_certs=False)

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_text = update.message.text
    search_results = vectorstore.similarity_search(user_text, k=2)
    context_text = "\n".join([doc.page_content for doc in search_results]) # Ищем подходящий контекст в базе FAISS

    prompt = f"""Используй предоставленный контекст, чтобы ответить на вопрос пользователя.
    Если в контексте нет ответа, так и скажи, но вежливо.

    Контекст:
    {context_text}

    Вопрос: {user_text}
    """

    # Получение ответа от LLM тут
    response = llm.invoke(prompt)

    await update.message.reply_text(response.content)

async def main():
    app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()
    app.add_handler(MessageHandler(filters.TEXT & (~filters.COMMAND), handle_message))
    await app.run_polling()

/tmp/ipykernel_4633/1348086746.py:5: LangChainDeprecationWarning: The class `GigaChat` was deprecated in LangChain 0.3.5 and will be removed in 1.0. An updated version of the class exists in the `langchain-gigachat package and should be used instead. To use it run `pip install -U `langchain-gigachat` and import as `from `langchain_gigachat import GigaChat``.
  llm = GigaChat(credentials=GIGACHAT_CREDENTIALS, verify_ssl_certs=False)


In [10]:
if __name__ == '__main__': # Запуск
    asyncio.run(main())

RuntimeError: Cannot close a running event loop

В ячейке выше ошибок нет. Это обозначает, что бот был отключен. Все скриншоты демонстрации работы приложены